In [1]:
from pathlib import Path

NVMS_ROOT = Path("/home/jovyan/work-easi-eds/nvms_runs")
SCRATCH_ROOT = Path("/home/jovyan/scratch/eds/tiles")
COMPAT_ROOT = Path("/home/jovyan/scratch/eds/compat")
OUT_CSV = Path("/home/jovyan/work-easi-eds/nvms_runs/run_availability_index.csv")

# Optional: count only clr products (recommended if you always run --sr-only-clr/--fc-only-clr)
ONLY_CLR = True


In [2]:
import re
import pandas as pd

PAT = re.compile(
    r"^(?P<prefix>[a-z0-9]+)_(?P<scene>p\d{3}r\d{3})_d(?P<start>\d{8})(?P<end>\d{8})_(?P<tag>[a-z0-9]+)$",
    re.IGNORECASE
)

def find_runnable_jobs(nvms_root: Path) -> pd.DataFrame:
    rows = []
    for run_dir in sorted(nvms_root.glob("run*")):
        if not run_dir.is_dir():
            continue
        for job_dir in sorted(run_dir.iterdir()):
            if not job_dir.is_dir():
                continue

            m = PAT.match(job_dir.name)
            if not m:
                continue

            rows.append({
                "run": run_dir.name,
                "job_dir": str(job_dir),
                "job_name": job_dir.name,
                "scene": m.group("scene").lower(),          # p089r078
                "start_yyyymmdd": m.group("start"),
                "end_yyyymmdd": m.group("end"),
                "start_iso": f"{m.group('start')[:4]}-{m.group('start')[4:6]}-{m.group('start')[6:]}",
                "end_iso": f"{m.group('end')[:4]}-{m.group('end')[4:6]}-{m.group('end')[6:]}",
                "tag": m.group("tag").lower(),
            })

    df = pd.DataFrame(rows).sort_values(["run", "scene", "start_yyyymmdd", "end_yyyymmdd"]).reset_index(drop=True)
    return df

jobs_df = find_runnable_jobs(NVMS_ROOT)
print("Detected runnable folders:", len(jobs_df))
jobs_df.head(10)


Detected runnable folders: 219


,run,job_dir,job_name,scene,start_yyyymmdd,end_yyyymmdd,start_iso,end_iso,tag
0,run1,/home/jovyan/work-easi-eds/nvms_runs/run1/lzol...,lzolre_p089r078_d2023030620231024_dlwm6,p089r078,20230306,20231024,2023-03-06,2023-10-24,dlwm6
1,run1,/home/jovyan/work-easi-eds/nvms_runs/run1/lzol...,lzolre_p089r084_d2023012520231024_dlwm6,p089r084,20230125,20231024,2023-01-25,2023-10-24,dlwm6
2,run1,/home/jovyan/work-easi-eds/nvms_runs/run1/lzol...,lzolre_p090r088_d2023010820231015_dlwm5,p090r088,20230108,20231015,2023-01-08,2023-10-15,dlwm5
3,run1,/home/jovyan/work-easi-eds/nvms_runs/run1/lzol...,lzolre_p090r090_d2023010820230727_dlwm5,p090r090,20230108,20230727,2023-01-08,2023-07-27,dlwm5
4,run1,/home/jovyan/work-easi-eds/nvms_runs/run1/lzol...,lzolre_p091r076_d2023030420231022_dlwm6,p091r076,20230304,20231022,2023-03-04,2023-10-22,dlwm6
5,run1,/home/jovyan/work-easi-eds/nvms_runs/run1/lzol...,lzolre_p091r087_d2023030420230928_dlwm5,p091r087,20230304,20230928,2023-03-04,2023-09-28,dlwm5
6,run1,/home/jovyan/work-easi-eds/nvms_runs/run1/lzol...,lzolre_p091r089_d2023011520230216_dlwm5,p091r089,20230115,20230216,2023-01-15,2023-02-16,dlwm5
7,run1,/home/jovyan/work-easi-eds/nvms_runs/run1/lzol...,lzolre_p091r090_d2023011520230904_dlwm5,p091r090,20230115,20230904,2023-01-15,2023-09-04,dlwm5
8,run1,/home/jovyan/work-easi-eds/nvms_runs/run1/lzol...,lzolre_p092r075_d2023041220231029_dlwm6,p092r075,20230412,20231029,2023-04-12,2023-10-29,dlwm6
9,run1,/home/jovyan/work-easi-eds/nvms_runs/run1/lzol...,lzolre_p092r084_d2023022420230928_dlwm5,p092r084,20230224,20230928,2023-02-24,2023-09-28,dlwm5


In [3]:
from datetime import datetime
from collections import defaultdict

DATE_RE = re.compile(r"_(\d{8})_")

def parse_date_from_filename(name: str):
    m = DATE_RE.search(name)
    if not m:
        return None
    return datetime.strptime(m.group(1), "%Y%m%d").date()

def scan_product(scene: str, product: str, only_clr: bool = True):
    """
    product: "sr" or "fc"
    Returns: count, min_date, max_date
    """
    root = SCRATCH_ROOT / scene / product
    if not root.exists():
        return 0, None, None

    dates = []
    for p in root.rglob("*.tif"):
        n = p.name.lower()
        if only_clr and "_clr" not in n:
            continue
        d = parse_date_from_filename(p.name)
        if d:
            dates.append(d)

    if not dates:
        return 0, None, None
    return len(dates), min(dates), max(dates)

def scan_fmask(scene: str):
    """
    Best-effort: look for a mask folder or fmask in filename.
    You can tighten this once we confirm how your fmask is stored.
    """
    scene_root = SCRATCH_ROOT / scene
    if not scene_root.exists():
        return 0, None, None

    hits = []
    for p in scene_root.rglob("*.tif"):
        n = p.name.lower()
        if "fmask" in n or "mask" in n:
            d = parse_date_from_filename(p.name)
            if d:
                hits.append(d)

    if not hits:
        return 0, None, None
    return len(hits), min(hits), max(hits)

# Build a tile availability index for all tiles found in jobs
unique_scenes = sorted(jobs_df["scene"].unique())
rows = []
for scene in unique_scenes:
    sr_n, sr_min, sr_max = scan_product(scene, "sr", only_clr=ONLY_CLR)
    fc_n, fc_min, fc_max = scan_product(scene, "fc", only_clr=ONLY_CLR)
    fm_n, fm_min, fm_max = scan_fmask(scene)

    rows.append({
        "scene": scene,
        "sr_tiles": sr_n,
        "sr_min": sr_min,
        "sr_max": sr_max,
        "fc_tiles": fc_n,
        "fc_min": fc_min,
        "fc_max": fc_max,
        "fmask_tiles": fm_n,
        "fmask_min": fm_min,
        "fmask_max": fm_max,
        "has_fmask": fm_n > 0,
    })

avail_df = pd.DataFrame(rows).sort_values("scene").reset_index(drop=True)
avail_df.head(25)


,scene,sr_tiles,sr_min,sr_max,fc_tiles,fc_min,fc_max,fmask_tiles,fmask_min,fmask_max,has_fmask
0,p089r078,189,2015-12-13,2025-11-14,189,2015-12-13,2025-11-14,189,2015-12-13,2025-11-14,True
1,p089r084,145,2015-12-13,2025-10-13,145,2015-12-13,2025-10-13,145,2015-12-13,2025-10-13,True
2,p090r088,13,2023-03-05,2023-10-15,13,2023-03-05,2023-10-15,13,2023-03-05,2023-10-15,True
3,p090r090,5,2023-01-16,2023-07-27,5,2023-01-16,2023-07-27,5,2023-01-16,2023-07-27,True
4,p091r076,23,2023-03-04,2023-10-22,23,2023-03-04,2023-10-22,23,2023-03-04,2023-10-22,True
5,p091r087,10,2023-03-04,2023-09-28,10,2023-03-04,2023-09-28,10,2023-03-04,2023-09-28,True
6,p091r089,3,2023-01-15,2023-02-16,3,2023-01-15,2023-02-16,3,2023-01-15,2023-02-16,True
7,p091r090,12,2023-01-15,2023-09-04,12,2023-01-15,2023-09-04,12,2023-01-15,2023-09-04,True
8,p092r075,21,2023-04-12,2023-10-29,21,2023-04-12,2023-10-29,21,2023-04-12,2023-10-29,True
9,p092r084,16,2023-02-24,2023-09-28,16,2023-02-24,2023-09-28,16,2023-02-24,2023-09-28,True


In [4]:
master_df = jobs_df.merge(avail_df, on="scene", how="left")

# Handy “quick status”
master_df["has_sr_100plus"] = master_df["sr_tiles"].fillna(0) >= 100
master_df["has_fc_100plus"] = master_df["fc_tiles"].fillna(0) >= 100
master_df["eligible_sr_fc_100plus"] = master_df["has_sr_100plus"] & master_df["has_fc_100plus"]

master_df.head(10)


,run,job_dir,job_name,scene,start_yyyymmdd,end_yyyymmdd,start_iso,end_iso,tag,sr_tiles,...,fc_tiles,fc_min,fc_max,fmask_tiles,fmask_min,fmask_max,has_fmask,has_sr_100plus,has_fc_100plus,eligible_sr_fc_100plus
0,run1,/home/jovyan/work-easi-eds/nvms_runs/run1/lzol...,lzolre_p089r078_d2023030620231024_dlwm6,p089r078,20230306,20231024,2023-03-06,2023-10-24,dlwm6,189,...,189,2015-12-13,2025-11-14,189,2015-12-13,2025-11-14,True,True,True,True
1,run1,/home/jovyan/work-easi-eds/nvms_runs/run1/lzol...,lzolre_p089r084_d2023012520231024_dlwm6,p089r084,20230125,20231024,2023-01-25,2023-10-24,dlwm6,145,...,145,2015-12-13,2025-10-13,145,2015-12-13,2025-10-13,True,True,True,True
2,run1,/home/jovyan/work-easi-eds/nvms_runs/run1/lzol...,lzolre_p090r088_d2023010820231015_dlwm5,p090r088,20230108,20231015,2023-01-08,2023-10-15,dlwm5,13,...,13,2023-03-05,2023-10-15,13,2023-03-05,2023-10-15,True,False,False,False
3,run1,/home/jovyan/work-easi-eds/nvms_runs/run1/lzol...,lzolre_p090r090_d2023010820230727_dlwm5,p090r090,20230108,20230727,2023-01-08,2023-07-27,dlwm5,5,...,5,2023-01-16,2023-07-27,5,2023-01-16,2023-07-27,True,False,False,False
4,run1,/home/jovyan/work-easi-eds/nvms_runs/run1/lzol...,lzolre_p091r076_d2023030420231022_dlwm6,p091r076,20230304,20231022,2023-03-04,2023-10-22,dlwm6,23,...,23,2023-03-04,2023-10-22,23,2023-03-04,2023-10-22,True,False,False,False
5,run1,/home/jovyan/work-easi-eds/nvms_runs/run1/lzol...,lzolre_p091r087_d2023030420230928_dlwm5,p091r087,20230304,20230928,2023-03-04,2023-09-28,dlwm5,10,...,10,2023-03-04,2023-09-28,10,2023-03-04,2023-09-28,True,False,False,False
6,run1,/home/jovyan/work-easi-eds/nvms_runs/run1/lzol...,lzolre_p091r089_d2023011520230216_dlwm5,p091r089,20230115,20230216,2023-01-15,2023-02-16,dlwm5,3,...,3,2023-01-15,2023-02-16,3,2023-01-15,2023-02-16,True,False,False,False
7,run1,/home/jovyan/work-easi-eds/nvms_runs/run1/lzol...,lzolre_p091r090_d2023011520230904_dlwm5,p091r090,20230115,20230904,2023-01-15,2023-09-04,dlwm5,12,...,12,2023-01-15,2023-09-04,12,2023-01-15,2023-09-04,True,False,False,False
8,run1,/home/jovyan/work-easi-eds/nvms_runs/run1/lzol...,lzolre_p092r075_d2023041220231029_dlwm6,p092r075,20230412,20231029,2023-04-12,2023-10-29,dlwm6,21,...,21,2023-04-12,2023-10-29,21,2023-04-12,2023-10-29,True,False,False,False
9,run1,/home/jovyan/work-easi-eds/nvms_runs/run1/lzol...,lzolre_p092r084_d2023022420230928_dlwm5,p092r084,20230224,20230928,2023-02-24,2023-09-28,dlwm5,16,...,16,2023-02-24,2023-09-28,16,2023-02-24,2023-09-28,True,False,False,False


In [5]:
master_df.to_csv(OUT_CSV, index=False)
print("Wrote:", OUT_CSV)


Wrote: /home/jovyan/work-easi-eds/nvms_runs/run_availability_index.csv


In [6]:
master_df[master_df["eligible_sr_fc_100plus"]][
    ["run","job_name","scene","start_iso","end_iso","sr_tiles","fc_tiles","sr_min","sr_max","fc_min","fc_max","has_fmask"]
].sort_values(["run","scene"])


,run,job_name,scene,start_iso,end_iso,sr_tiles,fc_tiles,sr_min,sr_max,fc_min,fc_max,has_fmask
0,run1,lzolre_p089r078_d2023030620231024_dlwm6,p089r078,2023-03-06,2023-10-24,189,189,2015-12-13,2025-11-14,2015-12-13,2025-11-14,True
1,run1,lzolre_p089r084_d2023012520231024_dlwm6,p089r084,2023-01-25,2023-10-24,145,145,2015-12-13,2025-10-13,2015-12-13,2025-10-13,True
19,run1,lzolre_p093r085_d2023011420230919_dlwm5,p093r085,2023-01-14,2023-09-19,177,177,2016-02-12,2026-01-06,2016-02-12,2026-01-06,True
21,run1,lzolre_p093r087_d2023011420231029_dlwm4,p093r087,2023-01-14,2023-10-29,103,103,2016-02-12,2026-01-06,2016-02-12,2026-01-06,True
22,run1,lzolre_p094r073_d2023012120231028_dlwm5,p094r073,2023-01-21,2023-10-28,194,194,2016-02-19,2025-12-12,2016-02-19,2025-12-12,True
23,run1,lzolre_p094r084_d2023022220230926_dlwm4,p094r084,2023-02-22,2023-09-26,198,198,2016-02-19,2026-01-05,2016-02-19,2026-01-05,True
24,run1,lzolre_p094r085_d2023022220230926_dlwm4,p094r085,2023-02-22,2023-09-26,178,178,2016-03-06,2026-01-05,2016-03-06,2026-01-05,True
25,run1,lzolre_p094r086_d2023022220230926_dlwm4,p094r086,2023-02-22,2023-09-26,133,133,2016-03-22,2026-01-05,2016-03-22,2026-01-05,True
26,run1,lzolre_p094r087_d2023022220231004_dlwm4,p094r087,2023-02-22,2023-10-04,106,106,2016-02-03,2026-01-05,2016-02-03,2026-01-05,True
27,run1,lzolre_p095r072_d2023040220231027_dlwm5,p095r072,2023-04-02,2023-10-27,159,159,2016-02-26,2025-12-19,2016-02-26,2025-12-19,True


In [7]:
master_df[(master_df["sr_tiles"] < 100) | (master_df["fc_tiles"] < 100)][
    ["run","scene","start_iso","end_iso","sr_tiles","fc_tiles","sr_min","sr_max","fc_min","fc_max"]
].sort_values(["sr_tiles","fc_tiles"])


,run,scene,start_iso,end_iso,sr_tiles,fc_tiles,sr_min,sr_max,fc_min,fc_max
32,run1,p095r083,2023-04-02,2023-10-27,0,0,None,None,None,None
33,run1,p095r084,2023-01-12,2023-10-27,0,0,None,None,None,None
34,run1,p095r085,2023-01-12,2023-10-27,0,0,None,None,None,None
35,run1,p095r086,2023-01-12,2023-10-27,0,0,None,None,None,None
36,run1,p096r070,2022-11-16,2023-10-10,0,0,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...
15,run1,p093r074,2023-04-12,2023-10-21,84,84,2016-02-12,2022-02-28,2016-02-12,2022-02-28
152,run2,p093r074,2023-04-12,2023-10-21,84,84,2016-02-12,2022-02-28,2016-02-12,2022-02-28
20,run1,p093r086,2023-01-14,2023-09-11,87,87,2016-02-12,2023-11-22,2016-02-12,2023-11-22
153,run2,p093r086,2023-01-14,2023-09-11,87,87,2016-02-12,2023-11-22,2016-02-12,2023-11-22


In [8]:
master_df[
    (
        (master_df["sr_tiles"] >= 10) & (master_df["sr_tiles"] < 100)
    ) | (
        (master_df["fc_tiles"] >= 10) & (master_df["fc_tiles"] < 100)
    )
][
    ["run","scene","start_iso","end_iso","sr_tiles","fc_tiles","sr_min","sr_max","fc_min","fc_max"]
].sort_values(["sr_tiles","fc_tiles"])


,run,scene,start_iso,end_iso,sr_tiles,fc_tiles,sr_min,sr_max,fc_min,fc_max
5,run1,p091r087,2023-03-04,2023-09-28,10,10,2023-03-04,2023-09-28,2023-03-04,2023-09-28
12,run1,p092r087,2023-02-24,2023-10-30,10,10,2023-02-24,2023-10-30,2023-02-24,2023-10-30
147,run2,p091r087,2023-03-04,2023-09-28,10,10,2023-03-04,2023-09-28,2023-03-04,2023-09-28
11,run1,p092r086,2023-02-24,2023-09-28,11,11,2023-02-24,2023-09-28,2023-02-24,2023-09-28
150,run2,p092r086,2023-02-24,2023-09-28,11,11,2023-02-24,2023-09-28,2023-02-24,2023-09-28
7,run1,p091r090,2023-01-15,2023-09-04,12,12,2023-01-15,2023-09-04,2023-01-15,2023-09-04
148,run2,p091r090,2023-01-15,2023-09-04,12,12,2023-01-15,2023-09-04,2023-01-15,2023-09-04
2,run1,p090r088,2023-01-08,2023-10-15,13,13,2023-03-05,2023-10-15,2023-03-05,2023-10-15
10,run1,p092r085,2023-02-24,2023-10-06,13,13,2023-02-24,2023-10-06,2023-02-24,2023-10-06
149,run2,p092r085,2023-02-24,2023-10-06,13,13,2023-02-24,2023-10-06,2023-02-24,2023-10-06


In [9]:
from datetime import datetime

PIPELINE = "/home/jovyan/work-easi-eds/scripts/easi-scripts/eds_lsat_collection/ls89_fc_sr_pipeline.py"
TILE_SHP = "/home/jovyan/assets/eds_lsat_grid_min_max.shp"
SPAN_YEARS = 10

def iso_to_yyyymm(iso_date: str) -> str:
    """
    Convert YYYY-MM-DD → YYYYMM
    """
    return datetime.strptime(iso_date, "%Y-%m-%d").strftime("%Y%m")

def build_cmd(row) -> str:
    start_ym = iso_to_yyyymm(row["start_iso"])
    end_ym   = iso_to_yyyymm(row["end_iso"])

    return (
        f"python {PIPELINE} "
        f"--tile-shp {TILE_SHP} "
        f"--tile-id {row['scene']} "
        f"--span-years {SPAN_YEARS} "
        f"--run-download "
        f"--season-core-start {start_ym} "
        f"--season-core-end {end_ym}"
    )

# empty_tiles_df = master_df[
#     (master_df["sr_tiles"] == 0) &
#     (master_df["fc_tiles"] == 0)
# ].sort_values("scene")

# print(f"Tiles with 0 SR and 0 FC: {len(empty_tiles_df)}\n")

# for _, row in empty_tiles_df.iterrows():
#     print(f"# {row['scene']}  ({row['start_iso']} → {row['end_iso']})")
#     print(build_cmd(row))
#     print()

empty_tiles_df = (
    master_df[
        (master_df["sr_tiles"] == 0) &
        (master_df["fc_tiles"] == 0)
    ]
    .sort_values("scene")
    .drop_duplicates(subset=["scene", "start_iso", "end_iso"])
)

print(f"Tiles with 0 SR and 0 FC: {len(empty_tiles_df)}\n")

for _, row in empty_tiles_df.head(10).iterrows():
    print(f"# {row['scene']}  ({row['start_iso']} → {row['end_iso']})")
    print(build_cmd(row))
    print()


Tiles with 0 SR and 0 FC: 110

# p095r083  (2023-04-02 → 2023-10-27)
python /home/jovyan/work-easi-eds/scripts/easi-scripts/eds_lsat_collection/ls89_fc_sr_pipeline.py --tile-shp /home/jovyan/assets/eds_lsat_grid_min_max.shp --tile-id p095r083 --span-years 10 --run-download --season-core-start 202304 --season-core-end 202310

# p095r084  (2023-01-12 → 2023-10-27)
python /home/jovyan/work-easi-eds/scripts/easi-scripts/eds_lsat_collection/ls89_fc_sr_pipeline.py --tile-shp /home/jovyan/assets/eds_lsat_grid_min_max.shp --tile-id p095r084 --span-years 10 --run-download --season-core-start 202301 --season-core-end 202310

# p095r085  (2023-01-12 → 2023-10-27)
python /home/jovyan/work-easi-eds/scripts/easi-scripts/eds_lsat_collection/ls89_fc_sr_pipeline.py --tile-shp /home/jovyan/assets/eds_lsat_grid_min_max.shp --tile-id p095r085 --span-years 10 --run-download --season-core-start 202301 --season-core-end 202310

# p095r086  (2023-01-12 → 2023-10-27)
python /home/jovyan/work-easi-eds/scripts/e

In [10]:
COMPAT_ROOT = Path("/home/jovyan/scratch/eds/compat/files")

def compat_output_exists(row) -> bool:
    """
    Return True if compat outputs already exist for this scene/date window.
    """
    tile = scene_to_tile(row["scene"])
    start = iso_to_yyyymmdd(row["start_iso"])
    end   = iso_to_yyyymmdd(row["end_iso"])

    out_dir = COMPAT_ROOT / tile / f"{start}_{end}"

    # exists AND has files
    return out_dir.exists() and any(out_dir.iterdir())


In [11]:
from datetime import datetime
from pathlib import Path

# -----------------------------
# Config
# -----------------------------
PY = "/env/bin/python"
PIPE = "/home/jovyan/work-easi-eds/scripts/easi-scripts/eds-processing/easi_eds_master_processing_pipeline.py"
OUT = "/home/jovyan/scratch/eds/compat/files"

SR_ROOT = "/home/jovyan/scratch/eds/tiles"
FC_ROOT = "/home/jovyan/scratch/eds/tiles"

# Choose the timeseries product you are running (must match folder under compat/files/)
# Examples seen in your scan: "ndvi", "fc", "fpc"
TIMESERIES = "ndvi"

# This matches your scanning notebook root
COMPAT_ROOT = Path("/home/jovyan/scratch/eds/compat")
COMPAT_FILES_ROOT = COMPAT_ROOT / "files"

MIN_SR_TILES = 70
MIN_FC_TILES = 70

# Skip behaviour:
# - False: skip if ANY outputs exist for that scene under TIMESERIES (regardless of date window)
# - True : skip only if outputs exist for that exact date window under TIMESERIES
REQUIRE_DATE_WINDOW = True


# -----------------------------
# Helpers
# -----------------------------
def iso_to_yyyymmdd(iso_date: str) -> str:
    return datetime.strptime(iso_date, "%Y-%m-%d").strftime("%Y%m%d")


def scene_to_tile(scene: str) -> str:
    # p089r078 → 089_078
    return f"{scene[1:4]}_{scene[5:8]}"


def build_eds_cmd(row) -> str:
    tile = scene_to_tile(row["scene"])
    start = iso_to_yyyymmdd(row["start_iso"])
    end = iso_to_yyyymmdd(row["end_iso"])

    return f"""
{PY} "{PIPE}" \\
  --tile {tile} \\
  --start-date {start} \\
  --end-date {end} \\
  --sr-root {SR_ROOT} \\
  --fc-root {FC_ROOT} \\
  --out-root "{OUT}" \\
  --timeseries-source {TIMESERIES} \\
  --fc-only-clr \\
  --sr-only-clr \\
  --python-exe "{PY}" \\
  --diagnostics \\
  --force-compat
""".strip()


def compat_outputs_exist(row, require_date_window: bool = True) -> bool:
    """
    Returns True if compat outputs already exist for this scene for the selected TIMESERIES.

    It checks specifically under:
      /home/jovyan/scratch/eds/compat/files/<TIMESERIES>/<scene>/...

    If require_date_window=True, it also requires finding the start/end stamp.
    """
    scene = str(row["scene"])
    start = iso_to_yyyymmdd(row["start_iso"])
    end = iso_to_yyyymmdd(row["end_iso"])

    ts_root = COMPAT_FILES_ROOT / TIMESERIES
    if not ts_root.exists():
        return False

    scene_dir = ts_root / scene
    if not scene_dir.exists():
        return False

    # Scene exists under this TIMESERIES and has content
    if not any(scene_dir.rglob("*")):
        return False

    if not require_date_window:
        return True

    # Date-window-specific patterns (covering common naming)
    stamp_patterns = [
        f"*d{start}_{end}*",
        f"*{start}*{end}*",
    ]
    for pat in stamp_patterns:
        if any(scene_dir.rglob(pat)):
            return True

    return False


# -----------------------------
# Main: filter + print commands (skip existing)
# -----------------------------
ready_df = (
    master_df[
        (master_df["sr_tiles"] >= MIN_SR_TILES) &
        (master_df["fc_tiles"] >= MIN_FC_TILES)
    ]
    .sort_values(["scene", "start_iso", "end_iso"])
    .drop_duplicates(subset=["scene", "start_iso", "end_iso"])
)

print(
    f"Ready tiles (SR>={MIN_SR_TILES} & FC>={MIN_FC_TILES}) "
    f"for TIMESERIES='{TIMESERIES}': {len(ready_df)}\n"
)

printed = 0
skipped = 0

for _, row in ready_df.iterrows():
    if compat_outputs_exist(row, require_date_window=REQUIRE_DATE_WINDOW):
        skipped += 1
        print(
            f"# SKIP {row['scene']} ({row['start_iso']} → {row['end_iso']}) "
            f"— outputs exist for '{TIMESERIES}'"
        )
        continue

    printed += 1
    print(f"# RUN {row['scene']} ({row['start_iso']} → {row['end_iso']}) [{TIMESERIES}]")
    print(build_eds_cmd(row))
    print()

print(f"\nSummary: printed {printed} commands, skipped {skipped} (already exists).")


Ready tiles (SR>=70 & FC>=70) for TIMESERIES='ndvi': 22

# SKIP p089r078 (2023-03-06 → 2023-10-24) — outputs exist for 'ndvi'
# SKIP p089r084 (2023-01-25 → 2023-10-24) — outputs exist for 'ndvi'
# SKIP p093r074 (2023-04-12 → 2023-10-21) — outputs exist for 'ndvi'
# SKIP p093r081 (2023-03-19 → 2023-10-21) — outputs exist for 'ndvi'
# SKIP p093r083 (2023-03-19 → 2023-10-13) — outputs exist for 'ndvi'
# SKIP p093r084 (2023-03-19 → 2023-10-05) — outputs exist for 'ndvi'
# SKIP p093r085 (2023-01-14 → 2023-09-19) — outputs exist for 'ndvi'
# SKIP p093r086 (2023-01-14 → 2023-09-11) — outputs exist for 'ndvi'
# SKIP p093r087 (2023-01-14 → 2023-10-29) — outputs exist for 'ndvi'
# SKIP p094r073 (2023-01-21 → 2023-10-28) — outputs exist for 'ndvi'
# SKIP p094r084 (2023-02-22 → 2023-09-26) — outputs exist for 'ndvi'
# SKIP p094r085 (2023-02-22 → 2023-09-26) — outputs exist for 'ndvi'
# SKIP p094r086 (2023-02-22 → 2023-09-26) — outputs exist for 'ndvi'
# SKIP p094r087 (2023-02-22 → 2023-10-04) — ou

In [12]:
from datetime import datetime
from pathlib import Path

# -----------------------------
# Config
# -----------------------------
PY = "/env/bin/python"
PIPE = "/home/jovyan/work-easi-eds/scripts/easi-scripts/eds-processing/easi_eds_master_processing_pipeline.py"
OUT = "/home/jovyan/scratch/eds/compat/files"

SR_ROOT = "/home/jovyan/scratch/eds/tiles"
FC_ROOT = "/home/jovyan/scratch/eds/tiles"


TIMESERIES = "ndvi"

# NOTE: this matches your scanning notebook root
COMPAT_ROOT = Path("/home/jovyan/scratch/eds/compat")
COMPAT_FILES_ROOT = COMPAT_ROOT / "files"

MIN_SR_TILES = 70
MIN_FC_TILES = 70


# -----------------------------
# Helpers
# -----------------------------
def iso_to_yyyymmdd(iso_date: str) -> str:
    return datetime.strptime(iso_date, "%Y-%m-%d").strftime("%Y%m%d")


def scene_to_tile(scene: str) -> str:
    # p089r078 → 089_078
    return f"{scene[1:4]}_{scene[5:8]}"


def build_eds_cmd(row) -> str:
    tile = scene_to_tile(row["scene"])
    start = iso_to_yyyymmdd(row["start_iso"])
    end = iso_to_yyyymmdd(row["end_iso"])

    return f"""
{PY} "{PIPE}" \\
  --tile {tile} \\
  --start-date {start} \\
  --end-date {end} \\
  --sr-root {SR_ROOT} \\
  --fc-root {FC_ROOT} \\
  --out-root "{OUT}" \\
  --timeseries-source {TIMESERIES} \\
  --fc-only-clr \\
  --sr-only-clr \\
  --python-exe "{PY}" \\
  --diagnostics \\
  --force-compat
""".strip()


def compat_outputs_exist(row, require_date_window: bool = False) -> bool:
    """
    Returns True if compat outputs already exist for this scene.
    If require_date_window=True, it also looks for folders/files containing the start/end stamp.
    """
    scene = str(row["scene"])
    start = iso_to_yyyymmdd(row["start_iso"])
    end   = iso_to_yyyymmdd(row["end_iso"])

    if not COMPAT_FILES_ROOT.exists():
        return False

    # 1) Fast scene-level check: any directory named exactly the scene anywhere under files/
    #    e.g. files/fpc/p089r078 or files/ndvi/p089r078
    scene_dirs = list(COMPAT_FILES_ROOT.glob(f"**/{scene}"))
    if scene_dirs:
        if not require_date_window:
            # scene folder exists AND has content somewhere under it
            for d in scene_dirs:
                if d.is_dir() and any(d.rglob("*")):
                    return True
        else:
            # look for the date-stamped folder/file patterns under the scene directory
            # examples: shp_d20230306_20231024, *_d20230306_20231024*, etc.
            stamp_patterns = [
                f"*d{start}_{end}*",
                f"*{start}*{end}*",
            ]
            for d in scene_dirs:
                for pat in stamp_patterns:
                    if any(d.rglob(pat)):
                        return True

    # 2) Fallback: date-stamp anywhere under files/ for this scene
    #    (slower, but robust if structure varies)
    if require_date_window:
        for pat in [f"**/*{scene}*d{start}_{end}*", f"**/*{scene}*{start}*{end}*"]:
            if any(COMPAT_FILES_ROOT.glob(pat)):
                return True

    return False


# -----------------------------
# Main: filter + print commands (skip existing)
# -----------------------------
ready_df = (
    master_df[
        (master_df["sr_tiles"] >= MIN_SR_TILES) &
        (master_df["fc_tiles"] >= MIN_FC_TILES)
    ]
    .sort_values(["scene", "start_iso", "end_iso"])
    .drop_duplicates(subset=["scene", "start_iso", "end_iso"])
)

print(f"Ready tiles (SR>={MIN_SR_TILES} & FC>={MIN_FC_TILES}): {len(ready_df)}\n")

printed = 0
skipped = 0

# Set require_date_window=False to skip if ANY outputs exist for that scene,
# or True to only skip if outputs exist for that exact start/end window.
REQUIRE_DATE_WINDOW = True

for _, row in ready_df.iterrows():
    if compat_outputs_exist(row, require_date_window=REQUIRE_DATE_WINDOW):
        skipped += 1
        print(f"# SKIP {row['scene']} ({row['start_iso']} → {row['end_iso']}) — outputs exist")
        continue

    printed += 1
    print(f"# RUN {row['scene']} ({row['start_iso']} → {row['end_iso']})")
    print(build_eds_cmd(row))
    print()

print(f"\nSummary: printed {printed} commands, skipped {skipped} (already exists).")


Ready tiles (SR>=70 & FC>=70): 22

# SKIP p089r078 (2023-03-06 → 2023-10-24) — outputs exist
# SKIP p089r084 (2023-01-25 → 2023-10-24) — outputs exist
# SKIP p093r074 (2023-04-12 → 2023-10-21) — outputs exist
# SKIP p093r081 (2023-03-19 → 2023-10-21) — outputs exist
# SKIP p093r083 (2023-03-19 → 2023-10-13) — outputs exist
# SKIP p093r084 (2023-03-19 → 2023-10-05) — outputs exist
# SKIP p093r085 (2023-01-14 → 2023-09-19) — outputs exist
# SKIP p093r086 (2023-01-14 → 2023-09-11) — outputs exist
# SKIP p093r087 (2023-01-14 → 2023-10-29) — outputs exist
# SKIP p094r073 (2023-01-21 → 2023-10-28) — outputs exist
# SKIP p094r084 (2023-02-22 → 2023-09-26) — outputs exist
# SKIP p094r085 (2023-02-22 → 2023-09-26) — outputs exist
# SKIP p094r086 (2023-02-22 → 2023-09-26) — outputs exist
# SKIP p094r087 (2023-02-22 → 2023-10-04) — outputs exist
# SKIP p095r072 (2023-04-02 → 2023-10-27) — outputs exist
# SKIP p095r079 (2023-04-02 → 2023-10-27) — outputs exist
# RUN p095r080 (2023-04-02 → 2023-10-

In [16]:
from datetime import datetime
from pathlib import Path

# -----------------------------
# Config
# -----------------------------
PY = "/env/bin/python"
PIPE = "/home/jovyan/work-easi-eds/scripts/easi-scripts/eds-processing/easi_eds_master_processing_pipeline.py"
OUT = "/home/jovyan/scratch/eds/compat/files"

SR_ROOT = "/home/jovyan/scratch/eds/tiles"
FC_ROOT = "/home/jovyan/scratch/eds/tiles"

# Where compat outputs land (used to skip already-computed jobs)
# NOTE: This assumes outputs go under /compat/files/<tile>/<start>_<end>/...
COMPAT_ROOT = Path("/home/jovyan/scratch/eds/compat/files")

# Thresholds for “ready”
MIN_SR_TILES = 70
MIN_FC_TILES = 70


# -----------------------------
# Helpers
# -----------------------------
def iso_to_yyyymmdd(iso_date: str) -> str:
    """Convert YYYY-MM-DD → YYYYMMDD"""
    return datetime.strptime(iso_date, "%Y-%m-%d").strftime("%Y%m%d")


def scene_to_tile(scene: str) -> str:
    """p089r078 → 089_078"""
    scene = str(scene)
    if len(scene) < 8 or scene[0] != "p" or "r" not in scene:
        raise ValueError(f"Unexpected scene format: {scene!r}")
    return f"{scene[1:4]}_{scene[5:8]}"


def compat_out_dir(row) -> Path:
    """Build expected compat output directory for this row."""
    tile = scene_to_tile(row["scene"])
    start = iso_to_yyyymmdd(row["start_iso"])
    end = iso_to_yyyymmdd(row["end_iso"])
    return COMPAT_ROOT / tile / f"{start}_{end}"


def compat_output_exists(row) -> bool:
    """
    Return True if compat outputs already exist for this scene/date window.

    This is a conservative “exists + non-empty directory” check.
    If your pipeline outputs a different structure, adjust compat_out_dir().
    """
    out_dir = compat_out_dir(row)
    return out_dir.exists() and any(out_dir.iterdir())


def build_eds_cmd(row) -> str:
    """Build a bash-friendly multiline command for EDS master pipeline."""
    tile = scene_to_tile(row["scene"])
    start = iso_to_yyyymmdd(row["start_iso"])
    end = iso_to_yyyymmdd(row["end_iso"])

    return f"""
{PY} "{PIPE}" \\
  --tile {tile} \\
  --start-date {start} \\
  --end-date {end} \\
  --sr-root {SR_ROOT} \\
  --fc-root {FC_ROOT} \\
  --out-root "{OUT}" \\
  --timeseries-source fc \\
  --fc-only-clr \\
  --sr-only-clr \\
  --python-exe "{PY}" \\
  --diagnostics \\
  --force-compat
""".strip()


# -----------------------------
# Main: filter + print commands (skip existing)
# -----------------------------
# master_df is assumed to exist and have: scene, start_iso, end_iso, sr_tiles, fc_tiles

ready_df = (
    master_df[
        (master_df["sr_tiles"] >= MIN_SR_TILES) &
        (master_df["fc_tiles"] >= MIN_FC_TILES)
    ]
    .sort_values(["scene", "start_iso", "end_iso"])
    .drop_duplicates(subset=["scene", "start_iso", "end_iso"])  # prevents duplicate prompts
)

print(f"Ready tiles (SR>={MIN_SR_TILES} & FC>={MIN_FC_TILES}): {len(ready_df)}\n")

printed = 0
skipped = 0

for _, row in ready_df.iterrows():
    if compat_output_exists(row):
        skipped += 1
        out_dir = compat_out_dir(row)
        print(f"# SKIP {row['scene']} ({row['start_iso']} → {row['end_iso']}) — outputs exist: {out_dir}")
        continue

    printed += 1
    print(f"# RUN {row['scene']} ({row['start_iso']} → {row['end_iso']})")
    print(build_eds_cmd(row))
    print()

print(f"\nSummary: printed {printed} commands, skipped {skipped} (already exists).")


Ready tiles (SR>=70 & FC>=70): 14

# RUN p089r078 (2023-03-06 → 2023-10-24)
/env/bin/python "/home/jovyan/work-easi-eds/scripts/easi-scripts/eds-processing/easi_eds_master_processing_pipeline.py" \
  --tile 089_078 \
  --start-date 20230306 \
  --end-date 20231024 \
  --sr-root /home/jovyan/scratch/eds/tiles \
  --fc-root /home/jovyan/scratch/eds/tiles \
  --out-root "/home/jovyan/scratch/eds/compat/files" \
  --timeseries-source fc \
  --fc-only-clr \
  --sr-only-clr \
  --python-exe "/env/bin/python" \
  --diagnostics \
  --force-compat

# RUN p089r084 (2023-01-25 → 2023-10-24)
/env/bin/python "/home/jovyan/work-easi-eds/scripts/easi-scripts/eds-processing/easi_eds_master_processing_pipeline.py" \
  --tile 089_084 \
  --start-date 20230125 \
  --end-date 20231024 \
  --sr-root /home/jovyan/scratch/eds/tiles \
  --fc-root /home/jovyan/scratch/eds/tiles \
  --out-root "/home/jovyan/scratch/eds/compat/files" \
  --timeseries-source fc \
  --fc-only-clr \
  --sr-only-clr \
  --python-exe

In [27]:
from datetime import datetime
from pathlib import Path

PY = "/env/bin/python"
PIPE = "/home/jovyan/work-easi-eds/scripts/easi-scripts/eds-processing/easi_eds_master_processing_pipeline.py"
OUT = "/home/jovyan/scratch/eds/compat/files"

def iso_to_yyyymmdd(iso_date: str) -> str:
    return datetime.strptime(iso_date, "%Y-%m-%d").strftime("%Y%m%d")

def scene_to_tile(scene: str) -> str:
    # p089r078 → 089_078
    return f"{scene[1:4]}_{scene[5:8]}"

def build_eds_cmd(row) -> str:
    tile = scene_to_tile(row["scene"])
    start = iso_to_yyyymmdd(row["start_iso"])
    end = iso_to_yyyymmdd(row["end_iso"])

    return f"""

{PY} "{PIPE}" \\
  --tile {tile} \\
  --start-date {start} \\
  --end-date {end} \\
  --sr-root /home/jovyan/scratch/eds/tiles \\
  --fc-root /home/jovyan/scratch/eds/tiles \\
  --out-root "{OUT}" \\
  --timeseries-source fc \\
  --fc-only-clr \\
  --sr-only-clr \\
  --python-exe "{PY}" \\
  --diagnostics \\
  --force-compat
""".strip()

ready_df = master_df[
    (master_df["sr_tiles"] >= 70) &
    (master_df["fc_tiles"] >= 70)
].sort_values(["scene", "start_iso"])

print(f"Ready tiles (SR>=100 & FC>=100): {len(ready_df)}\n")

for _, row in ready_df.iterrows():
    print(build_eds_cmd(row))
    print()


Ready tiles (SR>=100 & FC>=100): 19

/env/bin/python "/home/jovyan/work-easi-eds/scripts/easi-scripts/eds-processing/easi_eds_master_processing_pipeline.py" \
  --tile 089_078 \
  --start-date 20230306 \
  --end-date 20231024 \
  --sr-root /home/jovyan/scratch/eds/tiles \
  --fc-root /home/jovyan/scratch/eds/tiles \
  --out-root "/home/jovyan/scratch/eds/compat/files" \
  --timeseries-source fc \
  --fc-only-clr \
  --sr-only-clr \
  --python-exe "/env/bin/python" \
  --diagnostics \
  --force-compat

/env/bin/python "/home/jovyan/work-easi-eds/scripts/easi-scripts/eds-processing/easi_eds_master_processing_pipeline.py" \
  --tile 089_078 \
  --start-date 20230306 \
  --end-date 20231024 \
  --sr-root /home/jovyan/scratch/eds/tiles \
  --fc-root /home/jovyan/scratch/eds/tiles \
  --out-root "/home/jovyan/scratch/eds/compat/files" \
  --timeseries-source fc \
  --fc-only-clr \
  --sr-only-clr \
  --python-exe "/env/bin/python" \
  --diagnostics \
  --force-compat

/env/bin/python "/home/

In [ ]:
from pathlib import Path
import re
import subprocess
import shlex
from datetime import datetime

# Where your run folders live (adjust if needed)
RUNS_ROOT = Path("/home/jovyan/work-easi-eds/nvms_runs/run1")   # <-- change if yours differs

PIPELINE = Path("/home/jovyan/work-easi-eds/scripts/easi-scripts/eds_lsat_collection/ls89_fc_sr_pipeline.py")
TILE_SHP = Path("/home/jovyan/assets/eds_lsat_grid_min_max.shp")

SPAN_YEARS = 10

# Toggle these
DRY_RUN = False          # matches your normal command
RUN_DOWNLOAD = True     # you said: pull the data for each run

# Where the pipeline typically writes downloaded tiles (adjust if your pipeline uses a different root)
# If you're unsure, keep the marker-only skip (still works).
DOWNLOAD_ROOT = Path("/home/jovyan/scratch/eds/tiles")

print("RUNS_ROOT:", RUNS_ROOT)
print("PIPELINE:", PIPELINE)
print("TILE_SHP:", TILE_SHP)
print("DOWNLOAD_ROOT:", DOWNLOAD_ROOT)


In [ ]:
RUN_RE = re.compile(
    r"""
    _(?P<scene>p\d{3}r\d{3})      # p089r078
    _d(?P<start>\d{8})(?P<end>\d{8})  # dYYYYMMDDYYYYMMDD
    """,
    re.VERBOSE | re.IGNORECASE,
)

def parse_run_folder(folder: Path):
    m = RUN_RE.search(folder.name)
    if not m:
        return None

    scene = m.group("scene").lower()          # p089r078
    start = m.group("start")                  # 20230306
    end = m.group("end")                      # 20231024

    # pipeline expects --tile-id p095r081 style
    tile_id = scene

    # your CLI example uses YYYY-MM-DD for download start/end
    start_iso = datetime.strptime(start, "%Y%m%d").strftime("%Y-%m-%d")
    end_iso = datetime.strptime(end, "%Y%m%d").strftime("%Y-%m-%d")

    return {
        "run_dir": folder,
        "tile_id": tile_id,
        "start": start,
        "end": end,
        "start_iso": start_iso,
        "end_iso": end_iso,
    }

# quick test: show what we detect
runs = []
for p in sorted(RUNS_ROOT.iterdir()):
    if p.is_dir():
        info = parse_run_folder(p)
        if info:
            runs.append(info)

print(f"Detected {len(runs)} runnable folders:")
for r in runs:
    print(" -", r["run_dir"].name, "=>", r["tile_id"], r["start_iso"], "to", r["end_iso"])


In [ ]:
def done_marker(run_dir: Path) -> Path:
    return run_dir / ".download.done"

def looks_downloaded(tile_id: str, start: str, end: str) -> bool:
    """
    Best-effort check: does DOWNLOAD_ROOT/<tile_id>/sr/... contain either date?
    This avoids rerunning downloads if data already exists.

    If your pipeline writes somewhere else, either:
      - change DOWNLOAD_ROOT, or
      - rely on the .download.done marker only.
    """
    scene_dir = DOWNLOAD_ROOT / tile_id
    if not scene_dir.exists():
        return False

    # Search for date strings in filenames under sr and fc (cheap-ish glob)
    # Keep it simple and robust.
    for sub in ("sr", "fc"):
        base = scene_dir / sub
        if not base.exists():
            continue

        # recursive glob for any tif containing the date
        if list(base.rglob(f"*{start}*.tif")):
            return True
        if list(base.rglob(f"*{end}*.tif")):
            return True

    return False


In [ ]:
from pathlib import Path
import pandas as pd
import re

DOWNLOAD_ROOT = Path("/home/jovyan/scratch/eds/tiles")  # adjust if needed

print("Scanning:", DOWNLOAD_ROOT)


In [ ]:
def count_tiles(tile_dir: Path, subdir: str) -> int:
    d = tile_dir / subdir
    if not d.exists():
        return 0
    return len(list(d.rglob("*.tif")))


In [ ]:
rows = []

for tile_dir in sorted(DOWNLOAD_ROOT.iterdir()):
    if not tile_dir.is_dir():
        continue

    tile_id = tile_dir.name.lower()

    sr_count = count_tiles(tile_dir, "sr")
    fc_count = count_tiles(tile_dir, "fc")
    fmask_count = count_tiles(tile_dir, "fmask")

    rows.append({
        "tile_id": tile_id,
        "sr_tiles": sr_count,
        "fc_tiles": fc_count,
        "fmask_tiles": fmask_count,
        "has_fmask": fmask_count > 0,
    })

df = pd.DataFrame(rows).sort_values("tile_id").reset_index(drop=True)
df


### -----------------------  DELETE specific or empty dirs ------------

In [ ]:
tiles_to_delete = df[
    (df.sr_tiles <= 50) & (df.fc_tiles <= 50)
]["tile_id"].tolist()

tiles_to_delete


In [ ]:
# ## manually specify tiles

# tiles_to_delete = [
#     "p090r078",
#     "p090r088",
#     "p091r084",
#     "p093r084",
# ]


In [ ]:
from pathlib import Path

DOWNLOAD_ROOT = Path("/home/jovyan/scratch/eds/tiles")
RUNS_ROOT = Path("/home/jovyan/work-easi-eds/nvms_runs/run1")

print("Tile data directories that would be deleted:\n")

for tile_id in tiles_to_delete:
    tile_path = DOWNLOAD_ROOT / tile_id
    if tile_path.exists():
        print("  DATA:", tile_path)
    else:
        print("  DATA: (missing)", tile_path)

print("\nRun marker files that would be deleted:\n")

for run_dir in RUNS_ROOT.iterdir():
    if not run_dir.is_dir():
        continue

    for tile_id in tiles_to_delete:
        if tile_id in run_dir.name:
            for marker in [".download.done", ".download.failed"]:
                m = run_dir / marker
                if m.exists():
                    print("  MARKER:", m)


In [ ]:
import shutil

confirm = input(
    "⚠️  This will permanently delete selected tile datasets AND run markers.\n"
    "Type YES to continue: "
)

if confirm != "YES":
    raise SystemExit("Aborted by user")

# Delete tile data
for tile_id in tiles_to_delete:
    tile_path = DOWNLOAD_ROOT / tile_id
    if tile_path.exists():
        print("Deleting tile data:", tile_path)
        shutil.rmtree(tile_path)
    else:
        print("Tile data already missing:", tile_path)

# Delete stale markers
for run_dir in RUNS_ROOT.iterdir():
    if not run_dir.is_dir():
        continue

    for tile_id in tiles_to_delete:
        if tile_id in run_dir.name:
            for marker in [".download.done", ".download.failed"]:
                m = run_dir / marker
                if m.exists():
                    print("Deleting marker:", m)
                    m.unlink()

print("\nCleanup complete.")


## -------------------------------- Pull the trigger ----------------------------------

In [ ]:
def build_cmd(tile_id: str, start_iso: str, end_iso: str) -> list[str]:
    cmd = [
        "python",
        str(PIPELINE),
        "--tile-shp", str(TILE_SHP),
        "--tile-id", tile_id,
        "--span-years", str(SPAN_YEARS),
    ]

    if RUN_DOWNLOAD:
        cmd.append("--run-download")

    # cmd += [
    #     "--download-start-date", start_iso,
    #     "--download-end-date", end_iso,
    # ]

    def yyyymm(iso_date: str) -> str:
        return iso_date.replace("-", "")[:6]

    cmd += [
        "--season-core-start", yyyymm(start_iso),
        "--season-core-end",   yyyymm(end_iso),
    ]

    if DRY_RUN:
        cmd.append("--dry-run")

    return cmd


ok = 0
skipped = 0
failed = 0

# Run on multiple tiles (2) at a time

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path
import subprocess, time, os, shutil

MAX_WORKERS = 2  # start safe
LOCK_ROOT = Path("/home/jovyan/scratch/eds/.tile_locks")
LOCK_ROOT.mkdir(parents=True, exist_ok=True)

def acquire_lock(lock_dir: Path, timeout_s: int = 6 * 60 * 60, poll_s: float = 2.0) -> None:
    start = time.time()
    while True:
        try:
            lock_dir.mkdir(parents=True, exist_ok=False)  # atomic
            (lock_dir / "pid.txt").write_text(str(os.getpid()))
            return
        except FileExistsError:
            if time.time() - start > timeout_s:
                raise TimeoutError(f"Timed out waiting for lock: {lock_dir}")
            time.sleep(poll_s)

def release_lock(lock_dir: Path) -> None:
    if lock_dir.exists():
        shutil.rmtree(lock_dir, ignore_errors=True)

def run_one_job(cmd, run_dir_str, tile_id, start, end, download_root_str):
    run_dir = Path(run_dir_str)
    download_root = Path(download_root_str)

    marker_done = run_dir / ".download.done"
    marker_fail = run_dir / ".download.failed"
    lock_dir = LOCK_ROOT / tile_id

    def looks_downloaded_local():
        scene_dir = download_root / tile_id
        if not scene_dir.exists():
            return False
        for sub in ("sr", "fc"):
            base = scene_dir / sub
            if not base.exists():
                continue
            if list(base.rglob(f"*{start}*.tif")) or list(base.rglob(f"*{end}*.tif")):
                return True
        return False

    # marker only counts if data exists (prevents stale-marker skipping)
    if marker_done.exists() and looks_downloaded_local():
        return ("skipped", run_dir.name, "marker+data")

    if looks_downloaded_local():
        marker_done.write_text("skipped: already present\n")
        return ("skipped", run_dir.name, "data exists")

    acquire_lock(lock_dir)
    try:
        proc = subprocess.run(cmd, text=True, capture_output=True)
        if proc.returncode != 0:
            marker_fail.write_text(proc.stderr + "\n")
            return ("failed", run_dir.name, proc.stderr[-4000:])
        marker_done.write_text("ok\n")
        return ("ok", run_dir.name, proc.stdout[-2000:])
    finally:
        release_lock(lock_dir)

# Build jobs from your runs
jobs = []
for r in runs:
    cmd = build_cmd(r["tile_id"], r["start_iso"], r["end_iso"])
    jobs.append((cmd, str(r["run_dir"]), r["tile_id"], r["start"], r["end"], str(DOWNLOAD_ROOT)))

print(f"Prepared {len(jobs)} jobs. Running with MAX_WORKERS={MAX_WORKERS} (tile-locked).")

ok = skipped = failed = 0

with ProcessPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = [ex.submit(run_one_job, *job) for job in jobs]
    for fut in as_completed(futures):
        status, name, msg = fut.result()
        if status == "ok":
            ok += 1
        elif status == "skipped":
            skipped += 1
        else:
            failed += 1
        print(f"[{status.upper():7}] {name} — {msg}")

print(f"\nDone. ok={ok}, skipped={skipped}, failed={failed}")


## Run on one tile at a time

In [ ]:


for r in runs:
    run_dir = r["run_dir"]
    tile_id = r["tile_id"]
    start = r["start"]
    end = r["end"]

    marker = done_marker(run_dir)

    # Skip rules
    if marker.exists():
        print(f"[SKIP] {run_dir.name} (marker exists)")
        skipped += 1
        continue

    if looks_downloaded(tile_id, start, end):
        print(f"[SKIP] {run_dir.name} (looks already downloaded under {DOWNLOAD_ROOT}/{tile_id})")
        # write marker so we don't keep rechecking
        marker.write_text("skipped: already present\n")
        skipped += 1
        continue

    cmd = build_cmd(tile_id, r["start_iso"], r["end_iso"])
    print("\n[RUN ]", run_dir.name)
    print("      ", " ".join(shlex.quote(c) for c in cmd))

    try:
        # run it (captures output so notebook doesn't explode)
        proc = subprocess.run(cmd, text=True, capture_output=True)
        print(proc.stdout)
        if proc.returncode != 0:
            print(proc.stderr)
            raise RuntimeError(f"returncode={proc.returncode}")

        # mark success
        marker.write_text("ok\n")
        ok += 1

    except Exception as e:
        failed += 1
        (run_dir / ".download.failed").write_text(str(e) + "\n")
        print(f"[FAIL] {run_dir.name}: {e}")

print(f"\nDone. ok={ok}, skipped={skipped}, failed={failed}")
